# Analytics Report & Insights Documentation — Supporting Notebook
**Data Science Consultant Internship — Task 5**

This notebook reproduces every statistic, chart, and finding cited in the stakeholder-facing *Analytics Report & Insights Documentation* (Breast Cancer Diagnostic Analytics). It is the technical evidence behind that report's executive summary, findings, and recommendations.

## 1. Business Objectives (context)
Business problem: give clinical operations teams a fast, interpretable way to triage likely-malignant cases from routine tumor measurements. This notebook answers: (1) which measurements most strongly indicate malignancy, (2) how reliably a simple model can classify cases, and (3) what triage rule follows from the data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

# Cleaned dataset carried forward from Task 3 (569 rows, 8 core features, no missing values/duplicates)
df = pd.read_csv('breast_cancer_clean.csv')
print(df.shape)
df.head()

## 2. Data & Key Findings
Reproduce the correlation and group-mean findings cited in Section 2 of the report.

In [ ]:
feature_cols = ['radius_mean','texture_mean','perimeter_mean','area_mean','smoothness_mean',
                'compactness_mean','concavity_mean','symmetry_mean']

df['diagnosis_binary'] = (df['diagnosis']=='benign').astype(int)
corr_target = df[feature_cols + ['diagnosis_binary']].corr()['diagnosis_binary'].drop('diagnosis_binary').sort_values()
print("Correlation with diagnosis (benign=1):")
print(corr_target.round(3))

print("\nMean feature values by diagnosis:")
print(df.groupby('diagnosis')[feature_cols].mean().round(2))

print("\nClass balance:")
print(df['diagnosis'].value_counts(normalize=True).round(3) * 100)

## 3. Visualizations
Each chart below matches the corresponding figure in the report.

In [ ]:
# 3.1 Case distribution by diagnosis
counts = df['diagnosis'].value_counts()
plt.figure(figsize=(6,5))
plt.bar(counts.index, counts.values, color=['#55A868','#C44E52'])
plt.title('Number of Cases by Diagnosis')
plt.ylabel('Count')
plt.show()

In [ ]:
# 3.2 Radius distribution by diagnosis
plt.figure(figsize=(7,5))
for d, c in {'benign':'#55A868','malignant':'#C44E52'}.items():
    sns.histplot(df[df['diagnosis']==d]['radius_mean'], bins=25, color=c, label=d, alpha=0.6)
plt.title('Distribution of Mean Radius by Diagnosis')
plt.xlabel('Radius Mean')
plt.legend()
plt.show()

In [ ]:
# 3.3 Radius vs concavity (combined signal)
plt.figure(figsize=(7,5.5))
for d, c in {'benign':'#55A868','malignant':'#C44E52'}.items():
    sub = df[df['diagnosis']==d]
    plt.scatter(sub['radius_mean'], sub['concavity_mean'], label=d, color=c, alpha=0.6, edgecolor='k')
plt.xlabel('Radius Mean'); plt.ylabel('Concavity Mean')
plt.title('Radius vs Concavity by Diagnosis')
plt.legend()
plt.show()

In [ ]:
# 3.4 Malignancy rate by tumor size category (via SQL, matches Task 4)
bins = [0, 12, 15, 18, 100]
labels = ['Small','Medium','Large','Very Large']
df['radius_category'] = pd.cut(df['radius_mean'], bins=bins, labels=labels)

conn = sqlite3.connect(':memory:')
df.to_sql('tumors', conn, index=False)
rate_df = pd.read_sql("""
    SELECT radius_category,
           ROUND(100.0 * SUM(CASE WHEN diagnosis='malignant' THEN 1 ELSE 0 END) / COUNT(*), 1) AS malignant_pct
    FROM tumors GROUP BY radius_category
""", conn)
conn.close()

order = ['Small','Medium','Large','Very Large']
rate_df['radius_category'] = pd.Categorical(rate_df['radius_category'], categories=order, ordered=True)
rate_df = rate_df.sort_values('radius_category')

plt.figure(figsize=(7,5))
plt.bar(rate_df['radius_category'], rate_df['malignant_pct'], color='#DD8452')
plt.ylabel('Malignant Rate (%)')
plt.title('Malignancy Rate by Tumor Size Category')
for i,v in enumerate(rate_df['malignant_pct']):
    plt.text(i, v+1.5, f"{v}%", ha='center', fontweight='bold')
plt.show()

## 4. Predictive Model (supports Section 3.5 and the Business Insights)

In [ ]:
X = df[feature_cols]
y = (df['diagnosis'] == 'malignant').astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred):.4f}")

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5.5,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['benign','malignant'], yticklabels=['benign','malignant'])
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Confusion Matrix - Logistic Regression')
plt.show()

## 5. Business Insights & Recommendations (summary)

- **Size is the strongest, most interpretable signal:** malignancy rate rises from 3.5% (Small) to 100% (Very Large) in near-perfect, monotonic fashion — easy to communicate to non-technical stakeholders.
- **The model adds value on top of simple rules:** 93% accuracy, 92.9% recall on malignant cases, with errors concentrated in the medium-size, genuinely ambiguous range.
- **Recommended triage rule:** flag by size first; route medium-size/borderline cases to the model or a human reviewer rather than auto-deciding on size alone.
- **Recommendation for deployment:** prioritize recall (catching true malignant cases) over raw accuracy, and revalidate this analysis on current patient data before any live clinical use — these findings are based on a fixed, historical 569-case reference dataset.

See the full stakeholder-facing version of these findings in `Analytics_Report_Insights_Documentation.docx`.